# Multi-day Pipeline V3

Same shape as `multi_day_pipeline.ipynb`, but the exit/return classification
is replaced with the **v3 spike-healed symmetric** logic from
`components.v3_pipeline_restructure.ipynb`.

For each `(date, system)` folder under `data/flight_data/`:

1. Load `flight_tracks.csv` + `detections.csv`.
2. Per-track: nullify frames with frame-to-frame speed > 7 m/s, linearly
   interpolate the gaps, centered rolling-mean (window 5) to smooth, drop
   tracks shorter than 10 valid frames.
3. Run **v3 exit** detection: speed (min < 0.5 m/s in head) + start position
   within 0.3 m of hive + backward extrapolation hits the 0.15 m hive sphere
   (auto-pass if already inside) + outward cosine ≥ 0.3 (skipped at hover speed).
4. Run **v3 return** detection (symmetric on the tail).
5. Also compute the legacy `hive_*_v1` columns so v1↔v3 can be diffed.
6. Compute per-track tortuosity (after smoothing).

Then aggregate exactly like the legacy multi-day pipeline:
- Greedy pairing of v3 exits → next un-matched v3 return per (date, system).
- Daily counts, return ratio, trip duration, median tortuosity, IFI-CV.
- Joins: `flower_visit_summary.csv`, `Power levels (dBm)*.csv`,
  `Data transfer*.csv`.
- Cycle / condition labels (BASELINE / ON / OFF, anchor = 2026-04-23).
- Sign-aligned indicators → `indicators_daily.csv`.

Writes everything under `data/multi_day_v3/` so it doesn't overwrite the legacy
multi_day outputs.


In [20]:
from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

# --- Date / system filter -------------------------------------------------
DATES      = pd.date_range("2026-06-02", "2026-06-03")              # None = all dates; or pd.date_range("2026-04-19","2026-05-09")
SYSTEM_IDS = [900, 939]

# --- Hive positions per camera -------------------------------------------
HIVE_BY_SYSTEM = {
    900: np.array([-0.04,  -0.665, -1.195], dtype=float),
    939: np.array([-0.086, -0.828, -1.045], dtype=float),
}

# --- Classifier windows & legacy v1 tolerance ----------------------------
TOL          = 0.10   # m — legacy v1 sphere radius (used ONLY for v1 booleans)
HEAD_FRAMES  = 10     # v3 inspects the first 10 frames for take-off
TAIL_FRAMES  = 10     # v3 inspects the last 10 frames for landing
FPS          = 60.0   # PATS-C frame rate

# --- v3 thresholds (mirror components.v3_pipeline_restructure.ipynb) -----
HIVE_SPHERE_V3            = 0.15    # m — extrapolation target sphere
SPEED_THRESHOLD_TAKEOFF   = 0.5
SPEED_THRESHOLD_LANDING   = 0.5
START_NEAR_HIVE_MAX       = 0.20    # m — exit:   first position ≤ this
END_NEAR_HIVE_MAX         = 0.30    # m — return: last position ≤ this
APPROACH_COS_MIN_EXIT     = 0.30
APPROACH_COS_MIN_RETURN   = 0.30

# --- v3 leniency knobs ---------------------------------------------------
USE_MIN_SPEED_FOR_THRESHOLD     = True   # use min speed in window, not mean
AUTO_PASS_EXTRAPOLATION_AT_HIVE = True   # already-at-hive auto-passes extrapolation
HOVER_SPEED_MIN                 = 0.10   # below this, skip cosine check
APPROACH_DIR_FRAME_OFFSET       = 10     # frames outside head/tail for direction

# --- Spike healing & smoothing -------------------------------------------
MAX_BIO_VELOCITY = 7.0
SMOOTH_WINDOW    = 5
MIN_TRACK_FRAMES = 10

# --- Cycle labels --------------------------------------------------------
CYCLE_ANCHOR = pd.Timestamp("2026-04-23")

# --- Paths ---------------------------------------------------------------
DATA_BASE = Path("../../../data/flight_data")
DATA_ROOT = Path("../../../data")          # parent of flight_data/ — used for dBm/xfer
OUT_DIR   = Path("data/multi_day")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Source: {DATA_BASE.resolve()}")
print(f"Output: {OUT_DIR.resolve()}")


Source: /Users/jaspe/Projects/Claude/Bumblebee-monitoring/data/flight_data
Output: /Users/jaspe/Projects/Claude/Bumblebee-monitoring/src/flight_analysis/pats/data/multi_day


In [21]:
# ---------------------------------------------------------------------------
# Helpers — spike healing, v3 classifiers, v1 classifiers, tortuosity
# ---------------------------------------------------------------------------

POS_COLS = ["posX_insect", "posY_insect", "posZ_insect"]


def heal_and_smooth_track(trk: pd.DataFrame) -> pd.DataFrame | None:
    """Per-track: heal velocity spikes, interpolate gaps, smooth with a
    centered rolling mean.  Returns None if < MIN_TRACK_FRAMES valid frames
    survive after cleaning.  Adds `is_interpolated` and `spike_flagged` cols."""
    trk = trk.copy().reset_index(drop=True)

    if "pos_valid_insect" in trk.columns:
        valid_mask = trk["pos_valid_insect"] == 1
    else:
        valid_mask = pd.Series(True, index=trk.index)

    coords = trk[POS_COLS].astype(float).copy()
    coords.loc[~valid_mask] = np.nan

    dt = 1.0 / FPS
    speed = np.linalg.norm(coords.diff().values, axis=1) / dt
    spike_mask = pd.Series(speed > MAX_BIO_VELOCITY, index=coords.index).fillna(False)
    coords.loc[spike_mask.values] = np.nan

    interpolated_mask = spike_mask.values | (~valid_mask.values)
    coords = coords.interpolate(method="linear", limit_direction="both")

    if coords.dropna().shape[0] < MIN_TRACK_FRAMES:
        return None

    smoothed = (coords
                .rolling(window=SMOOTH_WINDOW, center=True, min_periods=1)
                .mean())

    out = trk.copy()
    out[POS_COLS] = smoothed.values
    out["is_interpolated"] = interpolated_mask
    out["spike_flagged"]   = spike_mask.values
    return out


def line_passes_within_radius(point, direction, target, radius, require_forward=True):
    d_norm = np.linalg.norm(direction)
    if d_norm < 1e-9:
        return False, np.nan, float("inf")
    direction = direction / d_norm
    to_target = target - point
    t = float(np.dot(to_target, direction))
    closest_point = point + t * direction
    closest_dist  = float(np.linalg.norm(closest_point - target))
    if require_forward and t < 0:
        return False, t, closest_dist
    return (closest_dist <= radius), t, closest_dist


def _valid_coords_arr(trk):
    arr = trk[POS_COLS].to_numpy(dtype=float)
    mask = ~np.isnan(arr).any(axis=1)
    return arr[mask]


def classify_exit_v3(trk, hive):
    coords = _valid_coords_arr(trk)
    if len(coords) < HEAD_FRAMES:
        return None

    head = coords[:HEAD_FRAMES]
    dt = 1.0 / FPS

    head_velocities = np.diff(head, axis=0) / dt
    head_speeds     = np.linalg.norm(head_velocities, axis=1)
    initial_speed_mean = float(np.mean(head_speeds))
    initial_speed_min  = float(np.min(head_speeds))
    speed_for_threshold = (initial_speed_min if USE_MIN_SPEED_FOR_THRESHOLD
                           else initial_speed_mean)
    is_taking_off = speed_for_threshold < SPEED_THRESHOLD_TAKEOFF

    dir_end = min(HEAD_FRAMES + APPROACH_DIR_FRAME_OFFSET, len(coords))
    if APPROACH_DIR_FRAME_OFFSET > 0 and (dir_end - HEAD_FRAMES) >= 2:
        dir_velocities = np.diff(coords[HEAD_FRAMES:dir_end], axis=0) / dt
    else:
        dir_velocities = head_velocities
    mean_velocity = dir_velocities.mean(axis=0)
    speed_norm    = float(np.linalg.norm(mean_velocity))
    velocity_unit = mean_velocity / speed_norm if speed_norm >= 1e-9 else np.zeros(3)

    hits_sphere, t_close, dist_close = line_passes_within_radius(
        point=head[0], direction=-velocity_unit, target=hive,
        radius=HIVE_SPHERE_V3, require_forward=True,
    )
    start_dist_to_hive = float(np.linalg.norm(head[0] - hive))
    start_near_hive    = start_dist_to_hive <= START_NEAR_HIVE_MAX
    auto_passed = AUTO_PASS_EXTRAPOLATION_AT_HIVE and start_dist_to_hive <= HIVE_SPHERE_V3
    hits_sphere_eff = hits_sphere or auto_passed

    outward = head[0] - hive
    outward_norm = np.linalg.norm(outward)
    if outward_norm < 1e-9 or speed_norm < 1e-9:
        cos_outward = 0.0
    else:
        cos_outward = float(np.dot(velocity_unit, outward) / outward_norm)
    cos_check = (speed_norm < HOVER_SPEED_MIN) or (cos_outward >= APPROACH_COS_MIN_EXIT)

    return {
        "initial_speed_min": initial_speed_min,
        "start_dist":        start_dist_to_hive,
        "outward_cos":       cos_outward,
        "hive_exit_v3": bool(is_taking_off and hits_sphere_eff and start_near_hive and cos_check),
    }


def classify_return_v3(trk, hive):
    coords = _valid_coords_arr(trk)
    if len(coords) < TAIL_FRAMES:
        return None

    tail = coords[-TAIL_FRAMES:]
    dt = 1.0 / FPS

    tail_velocities = np.diff(tail, axis=0) / dt
    tail_speeds     = np.linalg.norm(tail_velocities, axis=1)
    terminal_speed_mean = float(np.mean(tail_speeds))
    terminal_speed_min  = float(np.min(tail_speeds))
    speed_for_threshold = (terminal_speed_min if USE_MIN_SPEED_FOR_THRESHOLD
                           else terminal_speed_mean)
    is_landing = speed_for_threshold < SPEED_THRESHOLD_LANDING

    n = len(coords)
    dir_start = max(n - TAIL_FRAMES - APPROACH_DIR_FRAME_OFFSET, 0)
    dir_end   = n - TAIL_FRAMES
    if APPROACH_DIR_FRAME_OFFSET > 0 and (dir_end - dir_start) >= 2:
        dir_velocities = np.diff(coords[dir_start:dir_end], axis=0) / dt
    else:
        dir_velocities = tail_velocities
    mean_velocity = dir_velocities.mean(axis=0)
    speed_norm    = float(np.linalg.norm(mean_velocity))
    velocity_unit = mean_velocity / speed_norm if speed_norm >= 1e-9 else np.zeros(3)

    hits_sphere, t_close, dist_close = line_passes_within_radius(
        point=tail[-1], direction=velocity_unit, target=hive,
        radius=HIVE_SPHERE_V3, require_forward=True,
    )
    end_dist_to_hive = float(np.linalg.norm(tail[-1] - hive))
    end_near_hive    = end_dist_to_hive <= END_NEAR_HIVE_MAX
    auto_passed = AUTO_PASS_EXTRAPOLATION_AT_HIVE and end_dist_to_hive <= HIVE_SPHERE_V3
    hits_sphere_eff = hits_sphere or auto_passed

    toward = hive - tail[-1]
    toward_norm = np.linalg.norm(toward)
    if toward_norm < 1e-9 or speed_norm < 1e-9:
        cos_toward = 0.0
    else:
        cos_toward = float(np.dot(velocity_unit, toward) / toward_norm)
    cos_check = (speed_norm < HOVER_SPEED_MIN) or (cos_toward >= APPROACH_COS_MIN_RETURN)

    return {
        "terminal_speed_min": terminal_speed_min,
        "end_dist":           end_dist_to_hive,
        "toward_cos":         cos_toward,
        "hive_return_v3": bool(is_landing and hits_sphere_eff and end_near_hive and cos_check),
    }


def classify_exit_v1(coords, hive):
    head = coords[:HEAD_FRAMES]
    if len(head) == 0:
        return False, np.nan
    d = float(np.min(np.linalg.norm(head - hive, axis=1)))
    return (d <= TOL), d


def classify_return_v1(coords, hive):
    tail = coords[-TAIL_FRAMES:]
    if len(tail) == 0:
        return False, np.nan
    d = float(np.min(np.linalg.norm(tail - hive, axis=1)))
    return (d <= TOL), d


def path_tortuosity(xyz):
    """Arc length / end-to-end displacement. >= 1.0; 1.0 means perfect line."""
    if xyz is None or len(xyz) < 2:
        return np.nan
    arc = float(np.linalg.norm(np.diff(xyz, axis=0), axis=1).sum())
    end_to_end = float(np.linalg.norm(xyz[-1] - xyz[0]))
    return np.nan if end_to_end < 1e-6 else arc / end_to_end


In [22]:
# ---------------------------------------------------------------------------
# Iterate (date, system) folders → heal → classify → accumulate
# ---------------------------------------------------------------------------

folder_pat = re.compile(r"^(\d{4}-\d{2}-\d{2})_system_(\d+)$")

if DATES is None:
    target_dates = None
else:
    target_dates = set(pd.to_datetime(DATES).strftime("%Y-%m-%d"))

day_sys_folders = sorted(p for p in DATA_BASE.iterdir() if p.is_dir())

per_track_rows  = []
daily_rows      = []

for d in day_sys_folders:
    m = folder_pat.match(d.name)
    if not m:
        continue
    date_str, sys_str = m.groups()
    sys_id = int(sys_str)
    if sys_id not in SYSTEM_IDS:
        continue
    if target_dates is not None and date_str not in target_dates:
        continue
    if sys_id not in HIVE_BY_SYSTEM:
        print(f"  skipping {d.name} — no hive position for system {sys_id}")
        continue

    det_csv = d / "detections.csv"
    ft_csv  = d / "flight_tracks.csv"
    if not det_csv.exists() or not ft_csv.exists():
        print(f"  skipping {d.name} — missing detections.csv or flight_tracks.csv")
        continue

    dets = pd.read_csv(det_csv)
    dets["ts"] = pd.to_datetime(dets["datetime"], format="%Y%m%d_%H%M%S", errors="coerce")

    ft_cols_wanted = ["detection_uid", "pos_valid_insect",
                      "posX_insect", "posY_insect", "posZ_insect", "elapsed"]
    ft = pd.read_csv(ft_csv, usecols=lambda c: c in ft_cols_wanted)
    ft_by_uid = dict(tuple(ft.groupby("detection_uid")))

    hive = HIVE_BY_SYSTEM[sys_id]

    n_tracks = 0
    n_dropped = 0
    n_spike_tracks = 0
    n_v1_exit = n_v3_exit = 0
    n_v1_ret  = n_v3_ret  = 0
    exit_ts   = []
    return_ts = []

    for _, det in dets.iterrows():
        uid = int(det["uid"])
        trk = ft_by_uid.get(uid)
        if trk is None or len(trk) == 0:
            continue
        if "elapsed" in trk.columns:
            trk = trk.sort_values("elapsed")

        cleaned = heal_and_smooth_track(trk)
        if cleaned is None:
            n_dropped += 1
            continue
        n_tracks += 1
        if cleaned["spike_flagged"].any():
            n_spike_tracks += 1

        coords = _valid_coords_arr(cleaned)
        if len(coords) < HEAD_FRAMES:
            continue

        is_v1_exit, _ = classify_exit_v1(coords, hive)
        is_v1_ret,  _ = classify_return_v1(coords, hive)
        ex_v3 = classify_exit_v3(cleaned, hive)
        rt_v3 = classify_return_v3(cleaned, hive)
        if ex_v3 is None or rt_v3 is None:
            continue

        is_v3_exit = ex_v3["hive_exit_v3"]
        is_v3_ret  = rt_v3["hive_return_v3"]

        if is_v1_exit: n_v1_exit += 1
        if is_v3_exit:
            n_v3_exit += 1
            exit_ts.append(det["ts"])
        if is_v1_ret:  n_v1_ret += 1
        if is_v3_ret:
            n_v3_ret += 1
            return_ts.append(det["ts"])

        per_track_rows.append({
            "date":           date_str,
            "system_id":      sys_id,
            "uid":            uid,
            "ts":             det["ts"],
            "n_samples":      int(len(cleaned)),
            "n_spike_frames": int(cleaned["spike_flagged"].sum()),
            "hive_exit_v1":   bool(is_v1_exit),
            "hive_return_v1": bool(is_v1_ret),
            "hive_exit_v3":   bool(is_v3_exit),
            "hive_return_v3": bool(is_v3_ret),
            "initial_speed_min":  ex_v3["initial_speed_min"],
            "start_dist":         ex_v3["start_dist"],
            "outward_cos":        ex_v3["outward_cos"],
            "terminal_speed_min": rt_v3["terminal_speed_min"],
            "end_dist":           rt_v3["end_dist"],
            "toward_cos":         rt_v3["toward_cos"],
            "tortuosity":         path_tortuosity(coords),
        })

    # --- Greedy trip pairing (v3 exits → next un-matched v3 return) ----
    ex_sorted = sorted(t for t in exit_ts   if pd.notna(t))
    rt_sorted = sorted(t for t in return_ts if pd.notna(t))
    used  = [False] * len(rt_sorted)
    trips = []
    for et in ex_sorted:
        for j, rt in enumerate(rt_sorted):
            if used[j] or rt <= et:
                continue
            trips.append((rt - et).total_seconds())
            used[j] = True
            break
    median_trip = float(np.median(trips)) if trips else np.nan
    mean_trip   = float(np.mean(trips))   if trips else np.nan
    n_matched   = len(trips)

    daily_rows.append({
        "date":          date_str,
        "system_id":     sys_id,
        "n_tracks":      n_tracks,
        "n_dropped":     n_dropped,
        "n_spike_tracks": n_spike_tracks,
        "n_exit_v1":     n_v1_exit,
        "n_exit_v3":     n_v3_exit,
        "n_return_v1":   n_v1_ret,
        "n_return_v3":   n_v3_ret,
        "re_ratio_v3":   (n_v3_ret / n_v3_exit) if n_v3_exit > 0 else np.nan,
        "median_trip_s": median_trip,
        "mean_trip_s":   mean_trip,
        "n_matched":     n_matched,
    })

    print(f"  {d.name}: tracks={n_tracks}  v3_ex={n_v3_exit}  v3_ret={n_v3_ret}  trips={n_matched}")

per_track     = pd.DataFrame(per_track_rows)
daily_summary = pd.DataFrame(daily_rows)
print(f"\nDays processed: {len(daily_summary)}.  Total tracks: {len(per_track):,}")


  2026-06-02_system_900: tracks=237  v3_ex=18  v3_ret=16  trips=14

Days processed: 1.  Total tracks: 237


In [23]:
print

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [24]:
# ---------------------------------------------------------------------------
# Persist per-track + daily summary (these are inputs to everything below)
# ---------------------------------------------------------------------------

per_track.to_csv(OUT_DIR / "per_track_indicators.csv", index=False)
daily_summary.to_csv(OUT_DIR / "daily_summary.csv", index=False)
print(f"Wrote per_track_indicators.csv  ({len(per_track):,} rows)")
print(f"Wrote daily_summary.csv         ({len(daily_summary):,} rows)")


Wrote per_track_indicators.csv  (237 rows)
Wrote daily_summary.csv         (1 rows)


In [25]:
# ---------------------------------------------------------------------------
# Daily indicators: median tortuosity + IFI-CV → base `indicators` frame
# ---------------------------------------------------------------------------

tort = (per_track.groupby(["date", "system_id"])["tortuosity"]
                  .median()
                  .reset_index(name="path_tortuosity"))


def _ifi_cv(group):
    ts = group.loc[group["hive_exit_v3"], "ts"].sort_values()
    if len(ts) < 3:
        return np.nan
    iv = pd.to_datetime(ts).diff().dropna().dt.total_seconds()
    if iv.mean() == 0:
        return np.nan
    return float(iv.std() / iv.mean())


ifi = (per_track.groupby(["date", "system_id"])
                 .apply(_ifi_cv, include_groups=False)
                 .reset_index(name="ifi_cv"))

indicators = (daily_summary
              .merge(tort, on=["date", "system_id"], how="left")
              .merge(ifi,  on=["date", "system_id"], how="left"))

print(f"Base indicators frame: {indicators.shape}")
print(indicators[["date","system_id","n_exit_v3","n_return_v3","re_ratio_v3",
                  "path_tortuosity","ifi_cv"]].head().to_string(index=False))


Base indicators frame: (1, 15)
      date  system_id  n_exit_v3  n_return_v3  re_ratio_v3  path_tortuosity  ifi_cv
2026-06-02        900         18           16     0.888889          2.77243 1.75981


In [9]:
# ---------------------------------------------------------------------------
# Flower-visit join (optional — produced by flower_visit_pipeline.ipynb)
# ---------------------------------------------------------------------------

fv_path = OUT_DIR / "flower_visit_summary.csv"
legacy_fv_path = Path("data/multi_day/flower_visit_summary.csv")

# Prefer a v3-local one if present, otherwise fall back to the legacy file.
fv_use = fv_path if fv_path.exists() else (legacy_fv_path if legacy_fv_path.exists() else None)

if fv_use is not None:
    fv = pd.read_csv(fv_use)
    fv["date"] = pd.to_datetime(fv["date"]).dt.strftime("%Y-%m-%d")
    indicators["date"] = pd.to_datetime(indicators["date"]).dt.strftime("%Y-%m-%d")
    fv_cols = [c for c in ["date", "system_id", "mean_handling_time_s",
                            "n_distinct_flowers", "median_handling_time_s",
                            "n_visits", "revisit_rate"]
               if c in fv.columns]
    indicators = indicators.merge(fv[fv_cols], on=["date", "system_id"], how="left")
    print(f"Joined flower_visit_summary from {fv_use}  ({len(fv)} rows). "
          f"Indicators: {indicators.shape}")
else:
    print(f"!!! flower_visit_summary.csv not found.  Run flower_visit_pipeline.ipynb.")
    indicators["mean_handling_time_s"] = np.nan
    indicators["n_distinct_flowers"]   = np.nan


Joined flower_visit_summary from data/multi_day/flower_visit_summary.csv  (82 rows). Indicators: (82, 20)


In [10]:
# ---------------------------------------------------------------------------
# dBm signal strength + data-transfer join
# ---------------------------------------------------------------------------

SENSOR_TO_SYSTEM = {6: 900, 4: 939}   # PROVISIONAL — confirm

dbm_candidates  = sorted(DATA_ROOT.glob("Power levels (dBm)*.csv"))
xfer_candidates = sorted(DATA_ROOT.glob("Data transfer*.csv"))
print(f"dBm files found:           {[p.name for p in dbm_candidates]}")
print(f"data-transfer files found: {[p.name for p in xfer_candidates]}")

if dbm_candidates:
    dbm = pd.read_csv(dbm_candidates[-1])
    dbm["timestamp"] = pd.to_datetime(dbm["timestamp"])
    dbm["date"]      = dbm["timestamp"].dt.strftime("%Y-%m-%d")
    dbm["system_id"] = dbm["sensor"].map(SENSOR_TO_SYSTEM)

    dbm_daily = (dbm.dropna(subset=["system_id"])
                    .groupby(["date", "system_id"])
                    .agg(mean_dbm=("dBm","mean"),
                         max_dbm =("dBm","max"),
                         median_dbm=("dBm","median"))
                    .reset_index())
    dbm_daily["system_id"] = dbm_daily["system_id"].astype(int)
    indicators = indicators.merge(dbm_daily, on=["date","system_id"], how="left")
    print(f"Joined dBm: {dbm_daily.shape}")
else:
    indicators["mean_dbm"]   = np.nan
    indicators["max_dbm"]    = np.nan
    indicators["median_dbm"] = np.nan
    print("(no dBm file — columns left as NaN)")

if xfer_candidates:
    xfer = pd.read_csv(xfer_candidates[-1])
    xfer["timestamp"] = pd.to_datetime(xfer["timestamp"])
    xfer["date"]      = xfer["timestamp"].dt.strftime("%Y-%m-%d")
    xfer["mbps"] = (xfer["data"].astype(str).str.extract(r"([\d.]+)")[0].astype(float))
    xfer_daily = (xfer.groupby("date")
                      .agg(mean_mbps=("mbps","mean"),
                           max_mbps =("mbps","max"))
                      .reset_index())
    indicators = indicators.merge(xfer_daily, on=["date"], how="left")
    print(f"Joined data-transfer: {xfer_daily.shape}")
else:
    indicators["mean_mbps"] = np.nan
    indicators["max_mbps"]  = np.nan
    print("(no data-transfer file — columns left as NaN)")

print(f"\nWith dBm + transfer: {indicators.shape}")


dBm files found:           []
data-transfer files found: []
(no dBm file — columns left as NaN)
(no data-transfer file — columns left as NaN)

With dBm + transfer: (82, 25)


In [11]:
# ---------------------------------------------------------------------------
# Cycle / condition labels (anchor = CYCLE_ANCHOR, 3-day ON/OFF blocks)
# ---------------------------------------------------------------------------

def label_date(d):
    d = pd.Timestamp(d)
    if d < CYCLE_ANCHOR:
        return "BASELINE"
    return "ON" if ((d - CYCLE_ANCHOR).days // 3) % 2 == 0 else "OFF"


indicators["date_ts"]   = pd.to_datetime(indicators["date"])
indicators["condition"] = indicators["date_ts"].apply(label_date)
indicators["cycle"]     = ((indicators["date_ts"] - CYCLE_ANCHOR).dt.days // 6).clip(lower=-1)

print(indicators[["date","system_id","condition","cycle"]].head(8).to_string(index=False))


      date  system_id condition  cycle
2026-04-13        900  BASELINE     -1
2026-04-14        900  BASELINE     -1
2026-04-15        900  BASELINE     -1
2026-04-16        900  BASELINE     -1
2026-04-17        900  BASELINE     -1
2026-04-18        900  BASELINE     -1
2026-04-19        900  BASELINE     -1
2026-04-20        900  BASELINE     -1


In [12]:
# ---------------------------------------------------------------------------
# Sign-aligned indicators → indicators_daily.csv  (positive = impairment direction)
# ---------------------------------------------------------------------------

indicators["neg_exit_count"] = -indicators["n_exit_v3"].astype(float)
indicators["neg_re_ratio"]   = -indicators["re_ratio_v3"]

INDICATORS = [
    "neg_exit_count",
    "neg_re_ratio",
    "path_tortuosity",
    "ifi_cv",
    "mean_handling_time_s",
    "n_distinct_flowers",
]

out_cols = (["date", "system_id", "condition", "cycle"]
            + INDICATORS
            + ["n_exit_v3", "n_return_v3", "re_ratio_v3", "median_trip_s", "mean_trip_s",
               "n_matched", "n_visits", "mean_dbm", "max_dbm", "median_dbm",
               "mean_mbps", "max_mbps"])
out_cols = [c for c in out_cols if c in indicators.columns]
indicators_daily = (indicators[out_cols]
                    .sort_values(["system_id", "date"])
                    .reset_index(drop=True))

OUT_PATH = OUT_DIR / "indicators_daily.csv"
indicators_daily.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}  ({len(indicators_daily)} rows, {len(indicators_daily.columns)} cols)")
print()
print("=== indicators_daily.csv preview ===")
print(indicators_daily.head(6).to_string(index=False))
print()
print("Coverage by (system, condition):")
print(indicators_daily.groupby(["system_id", "condition"]).size().to_string())


Wrote data/multi_day_v3/indicators_daily.csv  (82 rows, 22 cols)

=== indicators_daily.csv preview ===
      date  system_id condition  cycle  neg_exit_count  neg_re_ratio  path_tortuosity   ifi_cv  mean_handling_time_s  n_distinct_flowers  n_exit_v3  n_return_v3  re_ratio_v3  median_trip_s  mean_trip_s  n_matched  n_visits  mean_dbm  max_dbm  median_dbm  mean_mbps  max_mbps
2026-04-13        900  BASELINE     -1           -18.0     -1.277778         2.458777 2.025071              3.532315                   6         18           23     1.277778         1086.5  1000.687500         16        30       NaN      NaN         NaN        NaN       NaN
2026-04-14        900  BASELINE     -1          -118.0     -0.771186         2.356138 2.682244              6.551293                   7        118           91     0.771186         6031.0  6993.956044         91        56       NaN      NaN         NaN        NaN       NaN
2026-04-15        900  BASELINE     -1          -143.0     -0.965035    

In [13]:
# ---------------------------------------------------------------------------
# Diagnostic: v1 vs v3 deltas across days (per (date, system))
# ---------------------------------------------------------------------------

if not daily_summary.empty:
    delta = daily_summary.assign(
        d_exit   = daily_summary["n_exit_v1"]   - daily_summary["n_exit_v3"],
        d_return = daily_summary["n_return_v1"] - daily_summary["n_return_v3"],
    )
    print("Per-day deltas (v1 - v3):")
    print(delta[["date","system_id","n_exit_v1","n_exit_v3","d_exit",
                 "n_return_v1","n_return_v3","d_return"]]
          .to_string(index=False))

    print()
    print("Totals across all days:")
    tot = daily_summary[["n_tracks","n_exit_v1","n_exit_v3",
                          "n_return_v1","n_return_v3","n_matched"]].sum()
    print(tot.to_string())


Per-day deltas (v1 - v3):
      date  system_id  n_exit_v1  n_exit_v3  d_exit  n_return_v1  n_return_v3  d_return
2026-04-13        900         18         18       0           51           23        28
2026-04-14        900        127        118       9          169           91        78
2026-04-15        900        163        143      20          233          138        95
2026-04-16        900         91         83       8          165           77        88
2026-04-17        900        160        137      23          250          144       106
2026-04-18        900        363        281      82          484          277       207
2026-04-19        900        133        111      22          226          136        90
2026-04-20        900        164        144      20          253          151       102
2026-04-21        900         81         77       4          156           70        86
2026-04-21        939         43         22      21            9           33       -24
2026-0